In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

outcome_matrix = pd.read_csv("eol_outcome_matrix.csv", index_col=0)
cost_vector    = pd.read_csv("eol_cost_vector.csv",    index_col=0, header=0)
cost_vector.columns = ['mean_cost_s']
unit_summary   = pd.read_csv("eol_unit_summary.csv")
df_temp        = pd.read_csv("eol_cleaned.csv", low_memory=False)

C_full_cost = cost_vector['mean_cost_s'].sum()

print(f"Outcome matrix:     {outcome_matrix.shape}")
print(f"Full sequence cost: {C_full_cost:.2f} s")

In [ ]:
all_failing = unit_summary[unit_summary['overall_outcome'] == 0]['unit_id'].tolist()

genuine_fail_units = []
abort_only_units   = []

for unit_id in all_failing:
    unit_steps    = df_temp[df_temp['unit_id'] == unit_id]
    has_step_fail = (unit_steps['result'] == 'FAIL').any()
    if has_step_fail:
        genuine_fail_units.append(unit_id)
    else:
        abort_only_units.append(unit_id)

failing_units = genuine_fail_units
N_fail        = len(failing_units)
THRESHOLD =85
delta         = THRESHOLD / 1_000_000

print(f"Genuine FAIL units:   {N_fail}")
print(f"ABORT-only excluded:  {len(abort_only_units)}")
print(f"Max allowed escapes: {delta * N_fail:.4f}")

In [ ]:
fail_matrix      = outcome_matrix.loc[outcome_matrix.index.isin(failing_units)].copy()
detection_matrix = (fail_matrix == 0).astype(float).fillna(0)

step_detection_count = detection_matrix.sum(axis=0)
diagnostic_steps     = step_detection_count[step_detection_count > 0].index.tolist()

costs = cost_vector.reindex(outcome_matrix.columns)['mean_cost_s'].fillna(0)

detection_sets = {
    step: set(detection_matrix.index[detection_matrix[step] == 1].tolist())
    for step in diagnostic_steps
}
step_costs = {step: max(costs.get(step, 0), 0.001) for step in diagnostic_steps}

print(f"Diagnostic steps: {len(diagnostic_steps)}")
print(f"Non-diagnostic:   {outcome_matrix.shape[1] - len(diagnostic_steps)}")

In [ ]:
def greedy_cover(max_escapes):
    uncovered    = set(failing_units)
    selected     = []
    already_used = set()

    while len(uncovered) > max_escapes:
        best_step, best_score, best_newly = None, -1, set()

        for step in diagnostic_steps:
            if step in already_used:
                continue
            newly = detection_sets[step] & uncovered
            if not newly:
                continue
            score = len(newly) / step_costs[step]
            if score > best_score:
                best_score, best_step, best_newly = score, step, newly

        if best_step is None:
            break

        already_used.add(best_step)
        selected.append(best_step)
        uncovered -= best_newly

    n_escaped   = len(uncovered)
    subset_cost = sum(costs.get(s, 0) for s in selected)
    return selected, n_escaped, subset_cost

print("Greedy cover function defined")

In [ ]:
from itertools import combinations

# Enumerate ALL subsets of diagnostic steps (2^14 = 16384, manageable)
print(f"Diagnostic steps: {len(diagnostic_steps)}")
print(f"Total possible subsets: {2**len(diagnostic_steps):,}")
print("Enumerating all subsets...")

all_subset_results = []

for r in range(1, len(diagnostic_steps) + 1):
    for subset in combinations(diagnostic_steps, r):
        subset = list(subset)
        subset_cost = sum(costs.get(s, 0) for s in subset)
        n_escaped   = sum(
            1 for u in failing_units
            if not any(u in detection_sets[s] for s in subset)
        )
        escape_risk     = n_escaped / N_fail
        cost_saving_pct = (C_full_cost - subset_cost) / C_full_cost * 100
        all_subset_results.append({
            'escape_risk':     round(escape_risk, 6),
            'cost_saving_pct': round(cost_saving_pct, 4),
            'steps':           len(subset)
        })

all_subsets_df = pd.DataFrame(all_subset_results)
print(f"Total subsets enumerated: {len(all_subsets_df):,}")

# Build Pareto front from sweep
sweep_points = list(range(0, N_fail + 1))
pareto_results = []
prev_steps = None

for max_esc in sweep_points:
    selected_sw, n_escaped, subset_cost = greedy_cover(max_esc)
    escape_risk     = n_escaped / N_fail
    cost_saving_pct = (C_full_cost - subset_cost) / C_full_cost * 100
    pareto_results.append({
        'max_allowed_escapes': max_esc,
        'escaped_units':       n_escaped,
        'escape_risk':         round(escape_risk, 6),
        'escape_threshold':    round(escape_risk * 1_000_000, 2),
        'steps_selected':      len(selected_sw),
        'subset_cost_s':       round(subset_cost, 4),
        'cost_saving_pct':     round(cost_saving_pct, 4),
        'selected_steps':      selected_sw
    })

pareto_df = pd.DataFrame(pareto_results)
print(f"Pareto sweep points: {len(pareto_df)}")

In [ ]:
# Zero-escape operating point (answers RQ1 formally, feeds RQ3)
zero_escape = pareto_df[pareto_df['escaped_units'] == 0]
op          = zero_escape.iloc[-1]   # max saving with 0 escapes
C_red       = op['selected_steps']

C_red_cost     = sum(costs.get(s, 0) for s in C_red)
cost_saving_s  = C_full_cost - C_red_cost
cost_saving_pct = cost_saving_s / C_full_cost * 100

print(f"\n{'='*55}")
print("OPERATING POINT — EOL RQ2 (zero escapes)")
print(f"{'='*55}")
print(f"Steps in Cred:    {len(C_red)} / {outcome_matrix.shape[1]}")
print(f"Cost saving:      {cost_saving_pct:.2f}%")
print(f"Escape risk:      {op['escape_risk']:.6f}")
print(f"\nCred steps:")
for i, s in enumerate(C_red, 1):
    print(f"  {i}. {s}")

pd.DataFrame({'test_id': C_red}).to_csv("eol_rq2_cred.csv", index=False)
pareto_df.drop(columns='selected_steps').to_csv("eol_rq2_pareto_results.csv", index=False)
print("\nSaved: eol_rq2_cred.csv, eol_rq2_pareto_results.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

np.random.seed(0)
jitter_x = np.random.uniform(-0.008, 0.008, len(all_subsets_df))
jitter_y = np.random.uniform(-0.3,   0.3,   len(all_subsets_df))

ax.scatter(
    all_subsets_df['escape_risk']     + jitter_x,
    all_subsets_df['cost_saving_pct'] + jitter_y,
    color='steelblue', s=15, alpha=0.25,
    zorder=2, label='Candidate subsets'
)

pareto_front = pareto_df.drop_duplicates(
    subset=['escape_risk','cost_saving_pct']
).sort_values('escape_risk')

ax.plot(
    pareto_front['escape_risk'],
    pareto_front['cost_saving_pct'],
    color='black', linewidth=2,
    marker='o', markersize=5,
    zorder=4, label='Approx. Pareto Front'
)

ax.axvline(
    x=THRESHOLD/1_000_000, color='red',
    linestyle='--', linewidth=1.5,
    label='Quality threshold'
)

# Operating point — marker only, no legend
op = pareto_df[pareto_df['escaped_units'] == 0].iloc[-1]
ax.scatter(
    [op['escape_risk']], [op['cost_saving_pct']],
    color='red', marker='*', s=300,
    zorder=5    # ← no label
)

ax.set_xlim(-0.02, 1.0)
ax.set_ylim(89, 102)
ax.set_xlabel('Escape Risk (ε)', fontsize=14, fontweight='bold')
ax.set_ylabel('Cost Saving (%)', fontsize=14, fontweight='bold')
ax.yaxis.set_major_locator(plt.MultipleLocator(1))
ax.xaxis.set_major_locator(plt.MultipleLocator(0.1))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3)
ax.tick_params(axis='both', labelsize=11)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('eol_rq2_pareto_final.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eol_rq2_pareto_final.png")

In [ ]:
op = pareto_df[pareto_df['escaped_units'] == 0].iloc[-1]
C_red = op['selected_steps']
pd.DataFrame({'test_id': C_red}).to_csv("eol_rq2_cred.csv", index=False)
pareto_df.drop(columns='selected_steps').to_csv("eol_rq2_pareto_results.csv", index=False)
print("Saved: eol_rq2_cred.csv")
print(f"Cred steps: {len(C_red)}")
print(f"Cost saving: {op['cost_saving_pct']:.2f}%")